tables used:

processed - products, customers, geoloc, sellers

agg - orders_delivery_details, order_cost_detils, order_payments, order_review_summary

In [1]:
import pandas as pd
import duckdb
import os

conn = duckdb.connect('../database/olist.db')

# Order delivery

In [22]:
query = """
    SELECT * 
    FROM orders_delivery_details
"""

result = conn.execute(query).df().head()
print(result)

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status  days_till_approval  days_till_shipped  days_till_delivery  \
0    delivered                   0                  2                   8   
1    delivered                   2                  2                  14   
2    delivered                   0                  0                   9   
3    delivered                   0                  4                  14   
4    delivered                   0                  1                   3   

   days_till_estim_delivery  delivery_accuracy  
0                        16                 -8  


In [23]:
# Average days till order is approved

# First let's check for outliers
query = """
    select 
        max(days_till_approval), avg(days_till_approval), median(days_till_approval) 
    from view_orders_delivered
"""
result = conn.execute(query).df().head()
print(result)

   max(days_till_approval)  avg(days_till_approval)  \
0                       31                  0.51273   

   median(days_till_approval)  
0                         0.0  


In [24]:
# Let's focus on delivered orders for now
query = """
    CREATE OR REPLACE VIEW view_orders_delivered AS
SELECT
    *
FROM orders_delivery_details
WHERE order_status = 'delivered'

"""
conn.execute(query)


In [31]:
# summary statistics
query = """
    select 
        max(days_till_approval) as max_approval_days,
        avg(days_till_approval) as avg_approval_days,
        median(days_till_approval) as median_approval_days,
        max(days_till_shipped) as max_shipping_days,
        avg(days_till_shipped) as avg_shipping_days,
        median(days_till_shipped) as median_shipping_days,
        max(days_till_delivery) as avg_delivery_days,
        avg(days_till_delivery) as avg_delivery_days,
        median(days_till_delivery) as avg_delivery_days,
        max(days_till_estim_delivery) as max_estim_del_days,
        avg(days_till_estim_delivery) as avg_estim_del_days,
        median(days_till_estim_delivery) as median_estim_del_days,
        max(delivery_accuracy) as max_delivery_accuracy,
        avg(delivery_accuracy) as avg_delivery_accuracy,
        median(delivery_accuracy) as median_delivery_accuracy
    from view_orders_delivered
"""
result = conn.execute(query).df().head()
print(result)

   max_approval_days  avg_approval_days  median_approval_days  \
0                 31            0.51273                   0.0   

   max_shipping_days  avg_shipping_days  median_shipping_days  \
0                126           3.213918                   2.0   

   avg_delivery_days  avg_delivery_days_1  avg_delivery_days_2  \
0                210            12.496849                 10.0   

   max_estim_del_days  avg_estim_del_days  median_estim_del_days  \
0                 156           24.372748                   24.0   

   max_delivery_accuracy  avg_delivery_accuracy  median_delivery_accuracy  
0                    188             -11.875889                     -12.0  


In [21]:
# Let's see which order took so long for approval
query = """
    select 
        *
    from view_orders_delivered
    where days_till_approval > 10

"""
result = conn.execute(query).df().head()
print(result)

                           order_id                       customer_id  \
0  6e57e23ecac1ae881286657694444267  2dda54e25d0984e12705c84d4030e6e0   
1  241adc087f5732067fc042dceb9cc6da  8bac337f299a513e8c90b9fe96b12dd1   
2  cf72398d0690f841271b695bbfda82d2  2b7fff075bda701552485ef3f0810257   
3  0c1426109d8295a688ee4182016bba59  d5114ea20c2a52447a40aa1e87bfd5cd   
4  1fab4ac9d85079b3da72a11475ae1685  f831c1fa80308975ec2b58e4877328e0   

  order_status  days_till_approval  days_till_shipped  days_till_delivery  \
0    delivered                  11                  5                   8   
1    delivered                  12                 23                  27   
2    delivered                  12                  3                  10   
3    delivered                  13                 14                  20   
4    delivered                  12                  3                   7   

   days_till_estim_delivery  delivery_accuracy  
0                        28                -20  


It seems like orders are being shipped before being approved. It is either a glitch in the system where the approval process was not triggered.

   max_approval_days  avg_approval_days  median_approval_days  \
0                 31            0.51273                   0.0   

   max_shipping_days  avg_shipping_days  median_shipping_days  \
0                126           3.213918                   2.0   

   avg_delivery_days  avg_delivery_days_1  avg_delivery_days_2  \
0                210            12.496849                 10.0   

   max_estim_del_days  avg_estim_del_days  median_estim_del_days  \
0                 156           24.372748                   24.0   

   max_delivery_accuracy  avg_delivery_accuracy  median_delivery_accuracy  
0                    188             -11.875889                     -12.0  
